# 03 - Prétraitement Audio & Extraction de Caractéristiques

**Projet :** Système de Synthèse Vocale pour l'Arabe Hassaniya par Apprentissage par Transfert
**Auteur :** Mohamed Salem Ebnou Echvagha Oubeid (C34613)
**Module :** Dialectes NLP — Master M1 IA
**Date :** Juin 2026

---

## Objectif

Dans ce notebook, nous prétraitons les données audio pour les préparer à l'entraînement du modèle TTS.
Les étapes clés sont :

1. **Extraire l'audio** du fichier parquet vers des fichiers WAV
2. **Rééchantillonner** à un taux standard (22050 Hz)
3. **Supprimer les silences** au début et à la fin
4. **Extraire les caractéristiques** : Mel spectrogrammes, MFCCs
5. **Valider** l'alignement audio-texte

### Pourquoi les Mel Spectrogrammes ?

Les modèles TTS modernes (Tacotron, VITS, etc.) utilisent les **Mel spectrogrammes** comme
représentation intermédiaire. Un Mel spectrogramme est une représentation temps-fréquence
qui imite la façon dont l'oreille humaine perçoit le son :

- **Axe X** : Temps
- **Axe Y** : Bandes de fréquences à l'échelle Mel (mettant en valeur les fréquences les mieux perçues par l'humain)
- **Intensité de couleur** : Énergie à chaque point temps-fréquence

## 1. Configuration

In [ ]:
# Installer les dépendances (pour Colab)
# !pip install librosa soundfile matplotlib pandas pyarrow numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import os
from tqdm import tqdm

import librosa
import librosa.display
import soundfile as sf

from IPython.display import Audio, display

# Configuration
SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 80
TOP_DB = 20  # pour la suppression des silences

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

print(f'Configuration :')
print(f'  Taux d\'échantillonnage : {SAMPLE_RATE} Hz')
print(f'  Taille FFT : {N_FFT}')
print(f'  Hop length : {HOP_LENGTH}')
print(f'  Bandes Mel : {N_MELS}')

In [ ]:
# Charger le jeu de données
df = pd.read_parquet('../audios_dataset.parquet')
print(f'{len(df)} échantillons chargés')

## 2. Extraction Audio & Rééchantillonnage

Nous extrayons les octets audio du fichier parquet, les décodons et les rééchantillonnons à
22050 Hz — le taux d'échantillonnage standard pour les modèles TTS comme Tacotron2 et VITS.

In [ ]:
def extract_audio(audio_dict, target_sr=SAMPLE_RATE):
    """Extraire et rééchantillonner l'audio depuis le dictionnaire du jeu de données."""
    audio_bytes = audio_dict.get('bytes', b'')
    if not audio_bytes:
        return None, None
    try:
        audio, sr = sf.read(io.BytesIO(audio_bytes))
        audio = audio.astype(np.float32)
        # Convertir stéréo en mono si nécessaire
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        # Rééchantillonner si nécessaire
        if sr != target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        return audio, target_sr
    except Exception as e:
        print(f'Erreur : {e}')
        return None, None

# Test avec le premier échantillon
audio, sr = extract_audio(df['audio'].iloc[0])
print(f'Test d\'extraction : forme={audio.shape}, sr={sr}, durée={len(audio)/sr:.2f}s')

In [ ]:
# Extraire tous les échantillons audio
audios = []
valid_indices = []
durations_before = []

for i in tqdm(range(len(df)), desc='Extraction audio'):
    audio, sr = extract_audio(df['audio'].iloc[i])
    if audio is not None and len(audio) > 0:
        audios.append(audio)
        valid_indices.append(i)
        durations_before.append(len(audio) / sr)

print(f'\nExtraction réussie : {len(audios)} / {len(df)} échantillons')
print(f'Durée totale : {sum(durations_before):.1f}s ({sum(durations_before)/60:.1f} min)')

## 3. Suppression des Silences

Les enregistrements audio contiennent souvent des silences au début et à la fin.
La suppression des silences aide le modèle TTS à se concentrer sur le contenu vocal réel.

In [ ]:
# Supprimer les silences de tous les échantillons
trimmed_audios = []
durations_after = []

for audio in tqdm(audios, desc='Suppression des silences'):
    trimmed, _ = librosa.effects.trim(audio, top_db=TOP_DB)
    trimmed_audios.append(trimmed)
    durations_after.append(len(trimmed) / SAMPLE_RATE)

time_saved = sum(durations_before) - sum(durations_after)
print(f'\nAvant suppression : {sum(durations_before):.1f}s au total')
print(f'Après suppression : {sum(durations_after):.1f}s au total')
print(f'Temps supprimé :    {time_saved:.1f}s ({100*time_saved/sum(durations_before):.1f}%)')

In [ ]:
# Visualiser avant/après suppression pour un échantillon
sample_idx = 0
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

t_before = np.linspace(0, len(audios[sample_idx])/SAMPLE_RATE, len(audios[sample_idx]))
axes[0].plot(t_before, audios[sample_idx], color='#1565C0', linewidth=0.5)
axes[0].set_title(f'Avant Suppression — Durée : {durations_before[sample_idx]:.2f}s')
axes[0].set_ylabel('Amplitude')

t_after = np.linspace(0, len(trimmed_audios[sample_idx])/SAMPLE_RATE, len(trimmed_audios[sample_idx]))
axes[1].plot(t_after, trimmed_audios[sample_idx], color='#2E7D32', linewidth=0.5)
axes[1].set_title(f'Après Suppression — Durée : {durations_after[sample_idx]:.2f}s')
axes[1].set_xlabel('Temps (s)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('../results/figures/silence_trimming_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Extraction de Caractéristiques : Mel Spectrogrammes

### Comment fonctionnent les Mel Spectrogrammes

1. La forme d'onde audio est découpée en trames superposées (fenêtres)
2. La FFT (Transformée de Fourier Rapide) est appliquée à chaque trame
3. Les bandes de fréquences sont regroupées en filtres à l'échelle Mel
4. Une compression logarithmique est appliquée (échelle dB)

Le résultat est une matrice 2D : **[n_mels × trames_temporelles]**

C'est le format d'entrée standard pour les modèles TTS modernes.

In [ ]:
def compute_mel_spectrogram(audio, sr=SAMPLE_RATE):
    """Calculer le Mel spectrogramme logarithmique à partir de la forme d'onde."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

def compute_mfcc(audio, sr=SAMPLE_RATE, n_mfcc=13):
    """Calculer les MFCCs (Coefficients Cepstraux à Fréquence Mel)."""
    return librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=n_mfcc,
        n_fft=N_FFT, hop_length=HOP_LENGTH
    )

# Calcul pour le premier échantillon
mel = compute_mel_spectrogram(trimmed_audios[0])
mfcc = compute_mfcc(trimmed_audios[0])
print(f'Forme du Mel spectrogramme : {mel.shape}  (n_mels × trames_temporelles)')
print(f'Forme des MFCC : {mfcc.shape}  (n_mfcc × trames_temporelles)')

In [ ]:
# Visualiser Mel spectrogramme, MFCC et forme d'onde pour plusieurs échantillons
fig, axes = plt.subplots(3, 3, figsize=(18, 12))

for col in range(3):
    audio = trimmed_audios[col]
    text = df['transcription'].iloc[valid_indices[col]]
    mel = compute_mel_spectrogram(audio)
    mfcc = compute_mfcc(audio)
    
    # Forme d'onde
    t = np.linspace(0, len(audio)/SAMPLE_RATE, len(audio))
    axes[0, col].plot(t, audio, color='#1565C0', linewidth=0.5)
    axes[0, col].set_title(f'Forme d\'onde — "{text[:30]}..."', fontsize=9)
    axes[0, col].set_ylabel('Amplitude')
    
    # Mel spectrogramme
    librosa.display.specshow(mel, y_axis='mel', x_axis='time',
                             sr=SAMPLE_RATE, hop_length=HOP_LENGTH, ax=axes[1, col])
    axes[1, col].set_title('Mel Spectrogramme', fontsize=9)
    
    # MFCC
    librosa.display.specshow(mfcc, x_axis='time', sr=SAMPLE_RATE,
                             hop_length=HOP_LENGTH, ax=axes[2, col])
    axes[2, col].set_title('MFCC', fontsize=9)
    axes[2, col].set_ylabel('Coefficient')

plt.tight_layout()
plt.savefig('../results/spectrograms/feature_extraction_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé dans results/spectrograms/feature_extraction_samples.png')

## 5. Extraction de Caractéristiques par Lots

In [ ]:
# Calculer les Mel spectrogrammes pour tous les échantillons
mel_specs = []
mel_lengths = []

for audio in tqdm(trimmed_audios, desc='Calcul des Mel spectrogrammes'):
    mel = compute_mel_spectrogram(audio)
    mel_specs.append(mel)
    mel_lengths.append(mel.shape[1])

print(f'\n{len(mel_specs)} Mel spectrogrammes calculés')
print(f'Longueur moyenne des spectrogrammes : {np.mean(mel_lengths):.0f} trames')
print(f'Min : {min(mel_lengths)} trames, Max : {max(mel_lengths)} trames')

In [ ]:
# Visualiser la distribution des longueurs de spectrogrammes
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(mel_lengths, bins=30, color='#FF5722', edgecolor='white', alpha=0.8)
ax.set_xlabel('Longueur du Spectrogramme (trames)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des Longueurs de Mel Spectrogrammes')
ax.axvline(np.mean(mel_lengths), color='red', linestyle='--',
           label=f'Moyenne : {np.mean(mel_lengths):.0f} trames')
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/spectrogram_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Sauvegarde des Données Traitées

In [ ]:
# Sauvegarder les fichiers audio traités en WAV
os.makedirs('../data/processed', exist_ok=True)

saved_count = 0
for i, (audio, idx) in enumerate(zip(trimmed_audios, valid_indices)):
    wav_path = f'../data/processed/hassaniya_{idx:04d}.wav'
    sf.write(wav_path, audio, SAMPLE_RATE)
    saved_count += 1

print(f'{saved_count} fichiers WAV traités sauvegardés dans data/processed/')

In [ ]:
# Sauvegarder les Mel spectrogrammes en tableaux numpy
os.makedirs('../data/processed/mels', exist_ok=True)

for i, (mel, idx) in enumerate(zip(mel_specs, valid_indices)):
    np.save(f'../data/processed/mels/hassaniya_{idx:04d}.npy', mel)

print(f'{len(mel_specs)} fichiers Mel spectrogrammes sauvegardés dans data/processed/mels/')

In [ ]:
# Écouter les échantillons traités
for i in range(min(3, len(trimmed_audios))):
    idx = valid_indices[i]
    text = df['transcription'].iloc[idx]
    print(f'\nÉchantillon {idx} : "{text}"')
    print(f'Durée : {durations_after[i]:.2f}s')
    display(Audio(trimmed_audios[i], rate=SAMPLE_RATE))

## 7. Résumé du Prétraitement

In [ ]:
print('=' * 60)
print('     RÉSUMÉ DU PRÉTRAITEMENT')
print('=' * 60)
print(f'  Échantillons traités :       {len(trimmed_audios)}')
print(f'  Taux d\'échantillonnage :     {SAMPLE_RATE} Hz')
print(f'  Bandes Mel spectrogramme :   {N_MELS}')
print(f'  Taille fenêtre FFT :         {N_FFT}')
print(f'  Hop length :                 {HOP_LENGTH}')
print(f'  Durée totale (après trim) :  {sum(durations_after):.1f}s')
print(f'  Durée moyenne :              {np.mean(durations_after):.2f}s')
print(f'  Longueur moy. spectrogramme : {np.mean(mel_lengths):.0f} trames')
print(f'  Silence supprimé :           {time_saved:.1f}s')
print('=' * 60)
print()
print('Fichiers de sortie :')
print('  - data/processed/*.wav         (audio nettoyé)')
print('  - data/processed/mels/*.npy    (Mel spectrogrammes)')
print()
print('Prêt pour la démo TTS (Notebook 04)')

## Conclusion

Nous avons prétraité avec succès tous les échantillons audio :

1. **Extraction** : Décodage des octets audio depuis le format parquet
2. **Rééchantillonnage** : Standardisation à 22050 Hz
3. **Suppression des silences** : Nettoyage des enregistrements
4. **Extraction de caractéristiques** : Calcul des Mel spectrogrammes et MFCCs
5. **Sauvegarde** : Fichiers WAV traités et Mel spectrogrammes sauvegardés

Les données prétraitées sont maintenant prêtes pour la démonstration du modèle TTS.

**Prochaine étape** : Notebook 04 — Démo du pipeline TTS avec modèle pré-entraîné.